In [2]:
import sys

!{sys.executable} -m pip install tensorflow

  Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.35.0-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached grpcio-1.80.0-cp312-cp312-win_amd64.whl.metadata (3.9 kB)
  Using cached keras-3.14.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached ml_dtypes-0.5.4-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached opt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 7.35.0 which is incompatible.


In [11]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
import random
# 1. Set Python core random seed
random.seed(42)

# 2. Set NumPy random seed (handles data shuffling/splitting variables)
np.random.seed(42)

# 3. Set TensorFlow random seed (handles weight initialization & dropout)
tf.random.set_seed(42)

# ==========================================
# 1. LOAD AND PREPROCESS DATA
# ==========================================
print("Loading dataset...")
df = pd.read_csv("water_potability.csv")

# Fill missing values with median (same as your original code)
# Separate the dataset by the target variable
df_potable = df[df["Potability"] == 1]
df_toxic = df[df["Potability"] == 0]

# Fill missing values using the median of their respective group
for col in df.columns:
    if col != "Potability":
        df.loc[df["Potability"] == 1, col] = df.loc[df["Potability"] == 1, col].fillna(df_potable[col].median())
        df.loc[df["Potability"] == 0, col] = df.loc[df["Potability"] == 0, col].fillna(df_toxic[col].median())


# Keep only rows that fall within 1.5 * IQR of the middle data
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

# This filters out rows that contain extreme outliers
df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

# Add these right after loading your data and handling missing values
df["Sulfate_Chloramine_Ratio"] = df["Sulfate"] / (df["Chloramines"] + 1e-5)
df["Hardness_pH_Interaction"] = df["Hardness"] * df["ph"]
df["TDS_Hardness_Ratio"] = df["Solids"] / (df["Hardness"] + 1e-5)

# Remember to update your X definition so it automatically includes these new columns!
X = df.drop("Potability", axis=1)
y = df["Potability"]
# Split data

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (Crucial for Neural Networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for later production use
joblib.dump(scaler, "ann_scaler.pkl")

# ==========================================
# 2. BUILD THE ANN ARCHITECTURE
# ==========================================
# Sequential model allows us to stack layers linearly
model = Sequential(
    [
        # Input Layer & First Hidden Layer: 64 neurons
        Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.3),  # Prevents overfitting by randomly dropping 30% of neurons
        # Second Hidden Layer: 32 neurons
        Dense(32, activation="relu"),
        Dropout(0.2),
        # Third Hidden Layer: 16 neurons
        Dense(16, activation="relu"),
        # Output Layer: 1 neuron with Sigmoid for binary classification (0 or 1)
        Dense(1, activation="sigmoid"),
    ]
)

# View the model summary
model.summary()

# ==========================================
# 3. COMPILE THE MODEL
# ==========================================
model.compile(
    optimizer="adam",  # Adaptive Moment Estimation (industry standard)
    loss="binary_crossentropy",  # Loss function for binary classification
    metrics=["accuracy"],
)

# ==========================================
# 4. TRAIN THE MODEL WITH EARLY STOPPING
# ==========================================
# Stop training automatically if the validation loss stops improving
early_stop = EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True
)

print("\nTraining the Artificial Neural Network...")
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,  # Maximum iterations
    batch_size=32,  # Process 32 samples at a time
    validation_split=0.2,  # Use 20% of training data for real-time validation
    callbacks=[early_stop],
    verbose=0,
)

# ==========================================
# 5. EVALUATE THE MODEL
# ==========================================
print("\nEvaluating model on test data...")
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"📊 ANN Test Accuracy: {accuracy * 100:.2f}%")

# Generate predictions (probabilities)
y_pred_prob = model.predict(X_test_scaled)
# Convert probabilities to binary outcomes (0 or 1) using a 0.5 threshold
y_pred = (y_pred_prob > 0.5).astype("int32")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred))

# Save the final trained ANN model
model.save("water_potability_ann.h5")
print("Model saved successfully as 'water_potability_ann.h5'!")

Loading dataset...


c:\Users\ACER\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_40 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,457 (13.50 KB)

 Trainable params: 3,457 (13.50 KB)

 Non-trainable params: 0 (0.00 B)


Training the Artificial Neural Network...

Evaluating model on test data...
📊 ANN Test Accuracy: 66.85%
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step



📋 Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.89      0.77       335
           1       0.62      0.29      0.40       199

    accuracy                           0.67       534
   macro avg       0.65      0.59      0.58       534
weighted avg       0.66      0.67      0.63       534

Model saved successfully as 'water_potability_ann.h5'!
